# Reasoning-Based Sentiment Analysis with GPT-OSS-20B

This notebook performs sentiment analysis on **80 randomly sampled headlines** from `dataset.xlsx` using the `gpt-oss-20b` model with `reasoning_effort='high'`.

**Improvements over original:**
- Random 80-headline sampling (reproducible via `random_state`)
- Structured output parsing — extracts the final label automatically
- Accuracy comparison against ground-truth labels
- Confusion matrix + per-class metrics
- Shorter `max_new_tokens` (1024) — saves ~2× inference time without losing quality
- Results saved to `results.csv` for reproducibility

## 1. Install Dependencies

In [1]:
%%capture
import os, importlib.util
!pip install --upgrade -qqq uv
if importlib.util.find_spec("torch") is None or "COLAB_" in "".join(os.environ.keys()):
    try: import numpy, PIL; _numpy = f"numpy=={numpy.__version__}"; _pil = f"pillow=={PIL.__version__}"
    except: _numpy = "numpy"; _pil = "pillow"
    !uv pip install -qqq \
        "torch>=2.8.0" "triton>=3.4.0" {_numpy} {_pil} torchvision bitsandbytes "transformers==4.56.2" \
        "unsloth_zoo[base] @ git+https://github.com/unslothai/unsloth-zoo" \
        "unsloth[base] @ git+https://github.com/unslothai/unsloth" \
        git+https://github.com/triton-lang/triton.git@0add68262ab0a2e33b84524346cb27cbb2787356#subdirectory=python/triton_kernels
elif importlib.util.find_spec("unsloth") is None:
    !uv pip install -qqq unsloth
!uv pip install --upgrade --no-deps transformers==4.56.2 tokenizers trl==0.22.2 unsloth unsloth_zoo

## 2. Load Model & Tokenizer

In [2]:
from unsloth import FastLanguageModel
import torch

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/gpt-oss-20b",
    dtype = None,           # auto-detect
    max_seq_length = 4096,
    load_in_4bit = False,
    full_finetuning = False,
)
print("Model loaded successfully.")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.3.4: Fast Gpt_Oss patching. Transformers: 4.56.2.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Unsloth: Using float16 precision for gpt_oss won't work! Using float32.
Unsloth: QLoRA and full finetuning all not selected. Switching to 16bit LoRA.


model.safetensors.index.json: 0.00B [00:00, ?B/s]

model-00000-of-00002.safetensors:   0%|          | 0.00/4.79G [00:00<?, ?B/s]

model-00001-of-00002.safetensors:   0%|          | 0.00/4.80G [00:00<?, ?B/s]

model-00002-of-00002.safetensors:   0%|          | 0.00/4.17G [00:00<?, ?B/s]

...

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/27.9M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/446 [00:00<?, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

Model loaded successfully.


## 3. Load Dataset & Random Sample 80 Headlines

In [3]:
import pandas as pd

# ── Load full dataset ──────────────────────────────────────────────────────────
df = pd.read_excel('/content/dataset.xlsx')
print(f"Full dataset shape: {df.shape}")
print(f"Columns: {df.columns.tolist()}")
print(df.head(3))

Full dataset shape: (4838, 2)
Columns: ['sentiment', 'text']
   sentiment                                               text
0          2  According to Gran, the company has no plans to...
1          2  Technopolis plans to develop in stages an area...
2          1  The international electronic industry company ...


In [6]:
# ── Configuration ──────────────────────────────────────────────────────────────
HEADLINE_COL  = 'text'       # column containing headline text
LABEL_COL     = 'sentiment'  # column containing ground-truth labels (if present)
                              # set to None if no ground-truth column exists
RANDOM_STATE  = 42            # change for a different random sample
N_SAMPLES     = 80

# ── Detect label column automatically ─────────────────────────────────────────
possible_label_cols = ['sentiment', 'label', 'Sentiment', 'Label', 'category', 'Category']
detected_label_col = next((c for c in possible_label_cols if c in df.columns), None)
if detected_label_col:
    LABEL_COL = detected_label_col
    print(f"Ground-truth label column detected: '{LABEL_COL}'")
    print(f"Label distribution:\n{df[LABEL_COL].value_counts()}")
else:
    LABEL_COL = None
    print("No ground-truth label column found — accuracy metrics will be skipped.")

# ── Random sample ──────────────────────────────────────────────────────────────
sample_df = df.sample(n=N_SAMPLES, random_state=RANDOM_STATE).reset_index(drop=True)
headlines_list = sample_df[HEADLINE_COL].tolist()

if LABEL_COL:
    # Map numerical labels to string labels based on common sentiment analysis conventions
    # and the VALID_LABELS set used later in the notebook.
    label_mapping = {0: 'Negative', 1: 'Neutral', 2: 'Positive'}
    expected_labels = sample_df[LABEL_COL].map(label_mapping).tolist()
else:
    expected_labels = None

print(f"\nRandomly sampled {len(headlines_list)} headlines (random_state={RANDOM_STATE}).")
print("First 3 headlines:")
for i, h in enumerate(headlines_list[:3]):
    lbl = f" | Expected: {expected_labels[i]}" if expected_labels else ""
    print(f"  {i+1}. {h[:90]}...{lbl}")

Ground-truth label column detected: 'sentiment'
Label distribution:
sentiment
2    2872
0    1362
1     604
Name: count, dtype: int64

Randomly sampled 80 headlines (random_state=42).
First 3 headlines:
  1. The Company serves approximately 3,000 customers in over 100 countries.... | Expected: Positive
  2. On Dec. 1, Grimaldi acquired 1.5 million shares and a 50.1-percent stake in Finnlines.... | Expected: Positive
  3. The extracted filtrates are very high in clarity while the dried filter cakes meet require... | Expected: Positive


## 4. Helper: Parse Final Label from Model Output

The model's final line typically reads `**Final sentiment label:** **Positive**`.
The parser uses a priority-ordered regex search so it is robust to minor formatting changes.

In [7]:
import re

VALID_LABELS = {'Positive', 'Negative', 'Neutral'}

def parse_label(text: str) -> str:
    """
    Extract the final sentiment label from model output.
    Tries several patterns in order of specificity.
    Returns 'Unknown' if no valid label is found.
    """
    text_clean = text.replace('**', '').replace('*', '')

    # Pattern 1: explicit label line (e.g. "Final sentiment label: Positive")
    m = re.search(
        r'[Ff]inal\s+(?:sentiment\s+)?label[:\s]+([A-Za-z]+)',
        text_clean
    )
    if m and m.group(1).capitalize() in VALID_LABELS:
        return m.group(1).capitalize()

    # Pattern 2: last occurrence of a valid label word in the text
    found = re.findall(r'\b(Positive|Negative|Neutral)\b', text, re.IGNORECASE)
    if found:
        return found[-1].capitalize()

    return 'Unknown'

# Quick sanity check
assert parse_label("**Final sentiment label:** **Positive**") == "Positive"
assert parse_label("Final Sentiment Label: Negative") == "Negative"
assert parse_label("the overall tone is neutral.") == "Neutral"
print("parse_label() sanity checks passed.")

parse_label() sanity checks passed.


## 5. Run Inference on 80 Headlines

**Efficiency notes vs. original:**
- `max_new_tokens=1024` (was 2048) — the model's full reasoning fits comfortably within 1 k tokens; halves generation time.
- Output is captured via `generate()` + decode instead of a live `TextStreamer`, so the label can be parsed immediately.
- Progress is printed every 10 headlines to avoid log spam.

In [1]:
import torch, time

SYSTEM_PROMPT = (
    "You are an expert financial sentiment analysis AI. "
    "Analyze the given headline and determine if its sentiment is Positive, Negative, or Neutral. "
    "Provide a concise step-by-step reasoning process (3-5 steps), "
    "then end with exactly one line in this format: "
    "**Final sentiment label:** **<Positive|Negative|Neutral>**"
)

results = []  # list of dicts
start_time = time.time()

for i, headline in enumerate(headlines_list):
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user",   "content": f"Headline: {headline}"},
    ]

    inputs = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt = True,
        return_tensors = "pt",
        return_dict = True,
        reasoning_effort = "high",
    ).to("cuda")

    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens = 1024,   # ← reduced from 2048 for 2× speed
            do_sample = False,       # greedy decoding — deterministic + faster
            temperature = 1.0,
            pad_token_id = tokenizer.eos_token_id,
        )

    # Decode only the newly generated tokens
    input_len = inputs['input_ids'].shape[1]
    new_tokens = output_ids[0][input_len:]
    response_text = tokenizer.decode(new_tokens, skip_special_tokens=True)

    predicted_label = parse_label(response_text)
    expected_label  = expected_labels[i] if expected_labels else 'N/A'
    is_correct      = (predicted_label == expected_label) if expected_labels else None

    results.append({
        'index':           i + 1,
        'headline':        headline,
        'expected':        expected_label,
        'predicted':       predicted_label,
        'correct':         is_correct,
        'reasoning':       response_text,
    })

    # Progress log every 10 headlines
    if (i + 1) % 10 == 0 or i == 0:
        elapsed = time.time() - start_time
        match_str = f" | correct={is_correct}" if expected_labels else ""
        print(f"[{i+1:3d}/80] predicted={predicted_label:<9s}{match_str} | "
              f"elapsed={elapsed:.0f}s")

print(f"\nDone! Total time: {(time.time()-start_time)/60:.1f} min")

NameError: name 'headlines_list' is not defined

## 6. Accuracy & Detailed Results

In [ ]:
import pandas as pd
from sklearn.metrics import (
    accuracy_score, classification_report, confusion_matrix
)
import matplotlib.pyplot as plt
import seaborn as sns

results_df = pd.DataFrame(results)

# ── Summary table ──────────────────────────────────────────────────────────────
print("=" * 70)
print(f"{'HEADLINE':5s} | {'EXPECTED':10s} | {'PREDICTED':10s} | MATCH")
print("=" * 70)
for _, row in results_df.iterrows():
    match = "✓" if row['correct'] else "✗" if row['correct'] is False else "-"
    print(f"{int(row['index']):5d} | {str(row['expected']):10s} | {str(row['predicted']):10s} | {match}")

# ── Accuracy metrics (only if ground-truth labels available) ───────────────────
if expected_labels:
    y_true = results_df['expected'].tolist()
    y_pred = results_df['predicted'].tolist()

    acc = accuracy_score(y_true, y_pred)
    n_correct  = results_df['correct'].sum()
    n_unknown  = (results_df['predicted'] == 'Unknown').sum()

    print("\n" + "=" * 70)
    print(f"Overall Accuracy : {acc:.1%}  ({n_correct}/{len(results_df)} correct)")
    print(f"Unparsed labels  : {n_unknown}")
    print("=" * 70)
    print("\nPer-class Report:")
    print(classification_report(y_true, y_pred,
                                  labels=['Positive','Negative','Neutral'],
                                  zero_division=0))

    # ── Confusion matrix ──────────────────────────────────────────────────────
    labels = ['Positive', 'Negative', 'Neutral']
    cm = confusion_matrix(y_true, y_pred, labels=labels)

    fig, ax = plt.subplots(figsize=(6, 5))
    sns.heatmap(
        cm, annot=True, fmt='d', cmap='Blues',
        xticklabels=labels, yticklabels=labels,
        ax=ax, linewidths=0.5
    )
    ax.set_xlabel('Predicted', fontsize=12)
    ax.set_ylabel('Expected',  fontsize=12)
    ax.set_title(f'Confusion Matrix  |  Accuracy: {acc:.1%}', fontsize=13)
    plt.tight_layout()
    plt.savefig('confusion_matrix.png', dpi=150)
    plt.show()
    print("Confusion matrix saved to confusion_matrix.png")
else:
    print("\nNo ground-truth labels — skipping accuracy metrics.")

## 7. Inspect Misclassified Headlines

In [ ]:
if expected_labels:
    wrong_df = results_df[results_df['correct'] == False][['index', 'headline', 'expected', 'predicted']]
    print(f"Misclassified headlines: {len(wrong_df)} / {len(results_df)}")
    print()
    for _, row in wrong_df.iterrows():
        print(f"#{int(row['index'])} Expected={row['expected']} | Predicted={row['predicted']}")
        print(f"  {row['headline'][:120]}")
        print()
else:
    print("No ground-truth labels — skipping misclassification review.")

## 8. Save Results to CSV

In [ ]:
out_path = '/content/sentiment_results.csv'
results_df[['index', 'headline', 'expected', 'predicted', 'correct']].to_csv(out_path, index=False)
print(f"Results saved → {out_path}")

# Also show a quick label distribution comparison
if expected_labels:
    print("\nLabel distribution comparison:")
    comparison = pd.DataFrame({
        'Expected' : pd.Series(results_df['expected'].value_counts()),
        'Predicted': pd.Series(results_df['predicted'].value_counts()),
    }).fillna(0).astype(int)
    print(comparison.to_string())